In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
from byte_util.util import single_to_double_float, all_sites
from byte_util.met_transport import update_physical_params
from byte_util.reaction import mineral_params, ssa_to_bsa
from min3p.input import InputFile
from min3p.output import write_min3p
import fsspec

rerun_spinup = True

def calcite_volfrac(caco3_pct, bulk_density, rho_calcite=2.71):
    return 0.0 if pd.isna(caco3_pct) else max(caco3_pct, 0)/100*bulk_density/rho_calcite

horizons = ['A', 'B', 'C']

# Get depth-varying intrinsic rate constants for CO2 production
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'

soil_pco2 = pd.read_csv(f'{s3_input_path}/soil_pco2.csv', index_col=0)
soil_phys = pd.read_parquet(f'{s3_input_path}/soil_physical_parameters.parquet')
soil_chem = pd.read_parquet(f'{s3_input_path}/soil_chemical_parameters.parquet')
root_params = pd.read_parquet(f'{s3_input_path}/root_water_parameters.parquet')

# Adjust soil_chem to remove low concentrations of calcite in the A horizon
soil_chem.loc[('Kuma', 'A'), 'caco3_lt_2_mm'] = 0.0
soil_chem.loc[('Palouse', 'A'), 'caco3_lt_2_mm'] = 0.0
soil_chem.loc[('Pullman', 'A'), 'caco3_lt_2_mm'] = 0.0
soil_chem.loc[('Yolo', 'A'), 'caco3_lt_2_mm'] = 0.0
soil_chem.loc[('Yolo', 'C'), 'caco3_lt_2_mm'] = 14.2593
soil_chem.loc[('Cecil', slice(None)), 'caco3_lt_2_mm'] = 0.0
soil_chem.loc[('Kalamazoo', ['A', 'B']), 'caco3_lt_2_mm'] = 0.0

In [ ]:
init_aqueous = pd.read_parquet('../simulations/met_forcing_rxn/min3p_runs/speciation/initial_aqueous_chem.parquet')
init_aqueous['equil_calcite'] = init_aqueous['equil_calcite'].astype(bool)

# Initialize sodium concentrations for Kalamazoo A/B to ensure they are >1e-5 mol/L
# (These will soon be forgotten over the 100-year spin-up)
init_aqueous.loc[('Kalamazoo', 'A'), 'na+1_mol.L'] *= 10
init_aqueous.loc[('Kalamazoo', 'B'), 'na+1_mol.L'] *= 10

In [ ]:
# Calculate initial surface flux for each site and simulation type
from byte_util.util import start_date
sim_types = ['spinup']

columns = ['site', 'flux'] + sim_types
init_forcing = pd.DataFrame(columns=columns, dtype=float)
init_forcing.set_index(['site', 'flux'], inplace=True)

for site in all_sites:
    forcing_file = f'{s3_input_path}/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Limit to 10-year simulation period
    end_date = pd.to_datetime(start_date) + pd.Timedelta(days=3651)
    met_forcing = met_forcing.loc[start_date:end_date, :]

    # Convert from mm/hr to m/s
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    # Convert transpiration from mm/hr to m/d
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24
    # Convert from m/d to 1/d
    dz = 0.01
    met_forcing['transpiration_factor'] = met_forcing['transpiration_m.d'] / dz

    for col in ['surface_flux_m.s', 'transpiration_factor']:
        # Initial forcing value for spinup and longterm is the long-term mean
        init_forcing.loc[(site, col), 'spinup'] = met_forcing[col].mean()

In [ ]:
from byte_util.reaction import co2_factors

basepath = Path('../simulations/met_forcing_rxn/min3p_runs/base')

# Load in and scale CO2 respiration
soil_resp_calc = 'GB94'
database_rate = 1e-13
co2_scaling_factor = np.array(list(co2_factors.values()), dtype=np.float64)
k_co2_path = f'{s3_input_path}/soilco2production_profiles_{soil_resp_calc}.npy'

with fsspec.open(k_co2_path, 'rb') as f:
    k_co2 = np.load(f)
k_co2 *= co2_scaling_factor[:, np.newaxis]

for site in all_sites:
    if not rerun_spinup:
        continue

    sitepath = Path(f'../simulations/met_forcing_rxn/min3p_runs/{site}')
    simpath = sitepath / 'spinup'
    shutil.rmtree(simpath, ignore_errors=True)
    simpath.mkdir(parents=True)

    infile = InputFile.load('spinup.dat', path=basepath)

    ## Update physical parameters
    update_physical_params(infile, site, soil_phys, root_params)

    ## Update vsflow boundary conditions
    # Adjust top boundary to initial rate
    bcvs = infile.boundary_conditions_vsflow
    top_boundary = bcvs.zones[0]
    infil_rate = init_forcing.loc[(site, 'surface_flux_m.s'), 'spinup']
    top_boundary.boundary_value = single_to_double_float(f'{infil_rate:0.3e}')
    # Adjust initial transpiration_factor
    for zone in infile.physical_parameters_vsflow.zones:
        rwu = zone.root_water_uptake
        trans_factor = init_forcing.loc[(site, 'transpiration_factor'), 'spinup']
        rwu.transpiration_factor = f'{trans_factor:0.6f}'

    ## Write initial mineral surface areas
    nz = infile.spatial_discretization.number_of_control_volumes_z
    zmin, zmax = infile.spatial_discretization.z_bounds
    dz = (zmax - zmin) / (nz - 1)
    minerals = ['x', 'y', 'z'] + infile.geochemical_system.minerals
    surface_areas = np.zeros((len(minerals), nz), dtype=float)
    # Add in z column
    surface_areas[minerals.index('z')] = np.linspace(zmin, zmax, nz)
    # For CO2_resp
    site_idx = all_sites.index(site)
    surface_areas[minerals.index('co2_resp')] = np.flip(k_co2[site_idx])/database_rate
    # for k-feldspar, montmorillonite, gibbsite and sio2(a,pt) calc initial surface area (m2/L soil)
    for mineral in ['k-feld-d-ph', 'na-montmor', 'sio2(a,pt)', 'gibbsite-ph', 'forst-ph']:
        surface_areas[minerals.index(mineral)] = ssa_to_bsa(**mineral_params[mineral])

    ### Adjust initial chemistry by horizon
    components = infile.geochemical_system.components
    icrt = infile.initial_conditions_reactive_transport
    a_bot = np.round(4.0 - soil_phys.loc[(site, 'A'), 'bottom_m'], 2)
    b_bot = np.round(4.0 - soil_phys.loc[(site, 'B'), 'bottom_m'], 2)
    # Yolo doesn't have a C horizon
    if site == 'Yolo':
        b_bot = a_bot
    # Pullman b horizon extends well beyond 5 m
    elif site == 'Pullman':
        b_bot = 0.00

    for hzn, zone in zip(horizons, icrt.zones):
        # Adjust initial pH
        ph = soil_chem.loc[(site, hzn), 'pH']
        zone.concentration_input.records[0].replace_content(f"{ph:0.2f}           'ph'")
        # Adjust initial al+3, sio2, co3-2, na+1, mg+2, ca+2, and k+1 based on speciation results
        for comp in ['co3-2', 'mg+2', 'ca+2', 'na+1', 'k+1', 'al+3', 'h4sio4', 'cl-1']:
            equil_calcite = init_aqueous.loc[(site, hzn), 'equil_calcite']
            if comp == 'co3-2' and not equil_calcite:
                continue
            conc = init_aqueous.loc[(site, hzn), f'{comp}_mol.L']
            conc_str = single_to_double_float(f"{conc:.4e}")
            idx = components.index(comp)
            if comp == 'cl-1':
                zone.concentration_input.records[idx].replace_content(f"{conc_str}      'charge'")
            else:
                zone.concentration_input.records[idx].replace_content(f"{conc_str}      'free'")

        # Adjust CEC and bulk density
        cec = soil_chem.loc[(site, hzn), 'CEC_meq_100g']
        rho = soil_phys.loc[(site, hzn), 'rho_g.cm3']
        zone.cation_exchange_capacity = f'{cec:0.2f}'
        zone.dry_bulk_density = f'{rho:0.2f}'
        zone.equilibrate_with_fixed_solution_composition = True

        # Add calcite
        ci = infile.geochemical_system.minerals.index('calcite-ph')
        caco3_pct = soil_chem.loc[(site, hzn), 'caco3_lt_2_mm']
        if caco3_pct < 0.01:
            calcite_phi = 1.0e-10
            calcite_str = '1.0d-10'
        else:
            calcite_phi = calcite_volfrac(caco3_pct, bulk_density=rho)
            calcite_phi = max(calcite_phi, 1e-10)
            calcite_str = single_to_double_float(f'{calcite_phi:.2e}')
        zone.mineral_input.records[2*ci].replace_content(f"{calcite_str:<9} .true.   'twothird-mix'")
        calcite_bsa = ssa_to_bsa(vol_frac=calcite_phi, **mineral_params['calcite-ph'])

        if hzn == 'A':
            zone.extent_of_zone = f'0.0 1.0  0.0 1.0  {a_bot:0.2f} 4.00'
            surface_areas[minerals.index('calcite-ph'), int(a_bot/dz)+1:] = calcite_bsa
        elif hzn == 'B':
            zone.extent_of_zone = f'0.0 1.0  0.0 1.0  {b_bot:0.2f} {a_bot:0.2f}'
            surface_areas[minerals.index('calcite-ph'), int(b_bot/dz)+1:int(a_bot/dz)+1] = calcite_bsa
        elif hzn == 'C':
            zone.extent_of_zone = f'0.0 1.0  0.0 1.0  0.00 {b_bot:0.2f}'
            surface_areas[minerals.index('calcite-ph'), :int(b_bot/dz)+1] = calcite_bsa

    # Delete C horizon for Pullman and B horizon for Yolo
    if site == 'Yolo':
        infile.delete_zone('B horizon chem', block_names=['initial_conditions_reactive_transport'])
    elif site == 'Pullman':
        infile.delete_zone('C horizon chem', block_names=['initial_conditions_reactive_transport'])

    ### Adjust influent chemistry
    bcrt = infile.boundary_conditions_reactive_transport

    # Adjust al and sio4 to be 2 orders of magnitude lower than mean soil concentrations
    # Influent base cation concentrations are 1/3 of mean soil concentrations
    influent_factors = {'mg+2': 0.33, 'ca+2': 0.33, 'na+1': 0.33, 'k+1': 0.33,
                        'al+3': 0.01, 'h4sio4':0.01}
    for comp, factor in influent_factors.items():
        all_conc = init_aqueous.loc[(site, horizons), f'{comp}_mol.L'].values
        # Because Kalamazoo soils are well-drained, we'll assume that farmers must apply
        # a higher base cation supply via irrigation
        if site == 'Kalamazoo' and comp in ['mg+2', 'ca+2', 'k+1', 'na+1']:
            factor *= 10.
        conc_str = single_to_double_float(f"{factor*np.nanmean(all_conc):.4e}")
        idx = components.index(comp)
        bcrt.zones[0].concentration_input.records[idx].replace_content(f"{conc_str}      'free'")

    infile.save(simpath / 'spinup.dat')

    # Write surface areas
    write_min3p(surface_areas, 'spinup.surf', minerals, folder=simpath,
                prefix='spinup', label='phi_i, T = initial')

    # Also copy over the *.rld file
    shutil.copy(basepath / 'spinup.rld', simpath / 'spinup.rld')